# Soft rules: annealing, the hardening gap, and purification

This notebook accompanies the docs page
[`soft-purification`](../../docs/examples/soft-purification.md). It follows the one solver in the
library that does not optimize the hard objective at all: `SoftVoronoiConfig` optimizes a
randomized rule by gradient descent and then hardens it. The docs page tells the story at a small
sample size; this notebook runs the full study across four problems, prints every table, and
re-renders the committed figure.

Set `SCOREQUANT_EXAMPLE_FAST=1` to shrink every sample and optimizer budget for a quick pass.

## The family being fitted

On a finite sample the hard objective is piecewise constant in a rule's parameters, so its
gradient is zero almost everywhere. The way out is to change the rule: a randomized quantizer
gives each score a distribution over cells, which is a legitimate decision rule with an ordinary
label law and therefore an ordinary Fisher information. `SoftVoronoiConfig` fits the softmax
family whose free parameters are the cell centers, initialized by weighted k-means and annealed
from the median nearest-center separation down to a fraction of it.

Double precision is an application-level choice. The library never sets it at import time, so the
notebook turns it on itself, before anything computes.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import scorequant as sq
from examples._env import example_scale
from examples.soft_purification import (
    SCHEDULE_PROBLEM,
    build_ladder,
    center_separation,
    fractional_retention,
    hard_retention,
    make_figure,
    run_study,
    softmax_responsibilities,
)

ladder = build_ladder()
traced = next(problem for problem in ladder if problem.key == SCHEDULE_PROBLEM)

[(problem.title, int(problem.scores.shape[1]), problem.n_bins) for problem in ladder]

## One annealed fit, traced

`diagnostics="full"` re-scores every recorded center snapshot with a complete information report.
That costs one full pass per snapshot, which is why it is not the default — and it is the only
way to watch the deployed rule while the soft objective climbs.

In [ ]:
steps = example_scale(300, 90)
rule = sq.fit_quantizer(
    sq.ScoreSample(traced.scores, traced.weights),
    n_bins=traced.n_bins,
    criterion=sq.DOptimality(),
    config=sq.SoftVoronoiConfig(
        seed=3,
        initializer_restarts=4,
        max_steps=steps,
        record_every=max(steps // 30, 1),
        temperature_end_ratio=0.02,
    ),
    diagnostics="full",
)
trace = rule.trace
temperatures = np.asarray(trace.temperatures)
soft = np.asarray(trace.soft_retention)
hard = np.asarray(trace.train_hard_retention)

print(f"randomized rule   {soft[0]:.6f} -> {soft[-1]:.6f}")
print(f"hard rule         {hard[0]:.6f} -> {hard[-1]:.6f}")
print(f"hardening gap     {rule.hardening_gap:+.3e}")

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(temperatures, soft, marker="o", markersize=3, label="randomized rule being optimized")
ax.plot(temperatures, hard, marker="s", markersize=3, label="hard rule the same centers imply")
ax.set(
    xscale="log",
    xlabel="temperature (annealed from left to right)",
    ylabel="D-efficiency",
    title=f"Annealing on {traced.title.lower()}",
)
ax.invert_xaxis()
ax.legend(loc="lower right");

The soft objective climbs a long way while the hard rule barely moves, and on this problem it
ends fractionally *below* the weighted k-means labeling it started from. Nothing is wrong: the
optimizer maximized the randomized objective at a nonzero temperature, and only in the
zero-temperature limit are the two the same function. It is a clean reason to judge a soft fit by
`train_hard_retention` and `hardening_gap` rather than by the objective history.

## The hardening gap across problems and temperatures

In [ ]:
study = run_study()
metrics = study.metrics
titles = {row["key"]: row["title"] for row in metrics["problems"]}
ratios = sorted({row["temperature_ratio"] for row in metrics["hardening"]}, reverse=True)

header = f"{'tau_end / tau_0':>16}" + "".join(f"{titles[key][:22]:>24}" for key in titles)
print(header)
print("-" * len(header))
for ratio in ratios:
    line = f"{ratio:>16.2f}"
    for key in titles:
        gap = next(
            row["hardening_gap"]
            for row in metrics["hardening"]
            if row["problem"] == key and row["temperature_ratio"] == ratio
        )
        line += f"{gap:>24.2e}"
    print(line)

Every entry above the level of floating-point noise is negative: hardening gained information
rather than costing it, and the gap closes by many orders of magnitude as the schedule cools.
Neither sign is guaranteed in general, which is exactly why the library reports the number
instead of assuming it.

## Purification, measured

If randomization could beat determinism the soft relaxation would be a genuinely larger class of
answers rather than a computational device. For an atomless population law it cannot: the
Dvoretzky–Wald–Wolfowitz elimination-of-randomization theorem replaces any randomized rule by a
deterministic one reproducing all cell masses and score moments exactly, and every criterion here
depends on a rule only through those moments. That is an existence statement about a population
law, and a finite sample is atomic, so what follows is a measurement rather than a proof.

`fractional_fisher_information` gives the randomized rule's information; the helper below turns it
into the same D-efficiency number `information_report` publishes. The way to trust that helper is
to check it against a one-hot responsibility matrix, where the two must agree exactly.

In [ ]:
labels = np.asarray(rule.predict_scores(traced.scores))
one_hot = np.eye(traced.n_bins)[labels]
print(
    "one-hot check:",
    fractional_retention(traced.scores, one_hot, traced.weights),
    hard_retention(traced.scores, labels, traced.weights, traced.n_bins),
)

coordinates = np.asarray(rule.transform.apply(traced.scores))
centers = np.asarray(rule.centers)
separation = center_separation(centers)

header = f"{'tau / separation':>18}{'randomized':>14}{'purified':>12}{'gain':>12}"
print()
print(header)
print("-" * len(header))
for ratio in (1.0, 0.5, 0.25, 0.1, 0.05):
    responsibilities = softmax_responsibilities(coordinates, centers, ratio * separation)
    randomized = fractional_retention(traced.scores, responsibilities, traced.weights)
    purified = hard_retention(
        traced.scores, np.argmax(responsibilities, axis=1), traced.weights, traced.n_bins
    )
    print(f"{ratio:>18.2f}{randomized:>14.6f}{purified:>12.6f}{purified - randomized:>12.2e}")

At a temperature comparable to the cell spacing the randomized rule is dramatically worse than
the same centers used deterministically, and the gap vanishes as the rule hardens. Across all
four problems and all five temperatures in the committed study the gain is positive everywhere.

## Against exact exchange

The last question is simply whether the soft path wins. Exact positive-gain exchange optimizes
the hard objective directly, and it can also be started from the soft fit's own labels.

In [ ]:
header = f"{'problem':<40}{'soft':>12}{'exchange':>12}{'from soft':>12}"
print(header)
print("-" * len(header))
for row in metrics["solvers"]:
    print(
        f"{titles[row['problem']]:<40}{row['soft_retention']:>12.7f}"
        f"{row['exchange_retention']:>12.7f}{row['exchange_from_soft_retention']:>12.7f}"
    )

## The committed figure

In [ ]:
figure = make_figure(study)
figure

## Interpretation

Exact exchange wins on retention on all four problems, by parts per million. Seeding exchange
with the soft labels recovers the exchange answer on some problems and stops slightly short on
others, because a soft solution is itself an exchange-stable point and a local search stops at the
first one it cannot improve. So the soft path is not the solver to reach for when a free labeling
of a fixed sample is what you want.

What it is, is the only route to two things exchange cannot do. It fits a *rule* rather than a
labeling, so it extends to events the fit never saw. And it accepts `ProfiledDOptimality`, whose
exchange result has no canonical compilation at all — the reusable profiled rule on the
[`nuisance-profiled-ds`](../../docs/examples/nuisance-profiled-ds.md) page is exactly this solver,
and there is no alternative to it.

Two practical habits follow. Cool the schedule: the hardening gap closes by orders of magnitude
between a warm and a cold final temperature, and a fit that stopped warm is reporting a rule
nobody optimized. And judge the fit by the hard retention of its recorded snapshots, never by the
soft objective, which can climb twenty-seven D-efficiency points while the deployed rule goes
nowhere at all.